## 4D synthetic loss panels (appendix figure)

`Run All` writes the 4 loss-dynamics panels into `img_results-2-x4/`:
`loss_p23_0.8_s23_0.0_p4_0.8_s4_{0.0,0.5,1.0,2.0}.pdf`, sweeping the noise level of the extra
spurious feature x4. The shared legend is drawn on the rightmost panel.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


def make_synthetic_dataset(n=10000, p_spurious_23=0.8, p_spurious_4=0.8,
                           sigma_23=0.0, sigma_4=0.0, seed=0):
    # 1 causal feature x1, two correlated spurious features x2/x3 (sign b23),
    # one independent spurious feature x4 (sign b4).
    rng = np.random.RandomState(seed)
    z = rng.uniform(-1.0, 1.0, size=n)
    x1_core = np.sin(5 * z) + np.exp(z)              # core feature; label = sign(x1_core)
    y = np.sign(x1_core)
    y01 = (y > 0).astype(np.float32)

    u = np.sin(5 * z)
    v = np.exp(z)
    b23 = rng.choice([-1.0, 1.0], size=n, p=[1 - p_spurious_23, p_spurious_23])
    b4 = rng.choice([-1.0, 1.0], size=n, p=[1 - p_spurious_4, p_spurious_4])

    x2 = np.empty(n, dtype=np.float32)
    x3 = np.empty(n, dtype=np.float32)
    x4 = np.empty(n, dtype=np.float32)
    for i in range(n):
        eps2 = rng.normal(0.0, sigma_23)
        eps3 = rng.normal(0.0, sigma_23)
        eps4 = rng.normal(0.0, sigma_4)
        x2[i] = b23[i] * u[i] + eps2
        x3[i] = b23[i] * v[i] + eps3
        x4[i] = b4[i] * (-2.0 * x1_core[i]) + eps4
    x1 = np.tanh(x1_core).astype(np.float32)         # causal feature shown to the model
    X = np.stack([x1, x2, x3, x4], axis=1).astype(np.float32)

    # near-boundary vs non-boundary by distance to the nearest opposite-class point in 4D
    pos_idx = np.where(y > 0)[0]
    neg_idx = np.where(y < 0)[0]
    X_pos = X[pos_idx]
    X_neg = X[neg_idx]
    min_opp_dist = np.empty(n, dtype=np.float32)
    for i in pos_idx:
        diff = X[i] - X_neg
        min_opp_dist[i] = np.sqrt(np.sum(diff * diff, axis=1)).min()
    for i in neg_idx:
        diff = X[i] - X_pos
        min_opp_dist[i] = np.sqrt(np.sum(diff * diff, axis=1)).min()
    is_confusing = min_opp_dist <= np.percentile(min_opp_dist, 20)
    is_nonconfusing = min_opp_dist >= np.percentile(min_opp_dist, 80)

    # spurious if EITHER sign(x2 + x3) OR sign(-x4) predicts y
    spur1_sign = np.sign(X[:, 1] + X[:, 2])
    spur1_sign[spur1_sign == 0] = 1
    spur2_sign = np.sign(-X[:, 3])
    spur2_sign[spur2_sign == 0] = 1
    spur_agrees_any = (spur1_sign == y) | (spur2_sign == y)

    groups = {
        "conf_spur": np.where(is_confusing & spur_agrees_any)[0],
        "conf_nonspur": np.where(is_confusing & ~spur_agrees_any)[0],
        "nonconf_spur": np.where(is_nonconfusing & spur_agrees_any)[0],
        "nonconf_nonspur": np.where(is_nonconfusing & ~spur_agrees_any)[0],
    }
    return X, y01, groups


class LogisticReg(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


def run_experiment(n=10000, p_spurious_23=0.8, p_spurious_4=0.8, sigma_23=0.0, sigma_4=0.0,
                   batch_size=256, num_epochs=10, lr=1e-1, seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    X, y01, groups = make_synthetic_dataset(
        n=n, p_spurious_23=p_spurious_23, p_spurious_4=p_spurious_4,
        sigma_23=sigma_23, sigma_4=sigma_4, seed=seed)

    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y01)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    model = LogisticReg(X.shape[1])
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    masks = {}
    for name, idxs in groups.items():
        mask = torch.zeros(X_t.size(0), dtype=torch.bool)
        mask[idxs] = True
        masks[name] = mask

    loss_history = {name: [] for name in groups}
    loss_history["all"] = []

    def record():
        with torch.no_grad():
            losses = criterion(model(X_t), y_t)
        loss_history["all"].append(losses.mean().item())
        for name, mask in masks.items():
            loss_history[name].append(losses[mask].mean().item() if mask.sum() > 0 else float("nan"))

    model.eval()
    record()
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in loader:
            loss = criterion(model(xb), yb).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        record()
    return loss_history


In [ ]:
import os
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

os.makedirs("img_results-2-x4", exist_ok=True)

BASE_P23 = 0.8
BASE_SIGMA23 = 0.0
p4 = 0.8
sigma4_list = [0.0, 0.5, 1.0, 2.0]   # noise level of the extra spurious feature x4

TICK_FONT, LABEL_FONT, LEGEND_FONT = 17, 19, 9
COLORS = {"conf_nonspur": "#3f8fd5", "conf_spur": "#ff9933",
          "nonconf_nonspur": "#009e73", "nonconf_spur": "#b07adf"}
LABELS = {"conf_nonspur": "Near-boundary & Non-spurious", "conf_spur": "Near-boundary & Spurious",
          "nonconf_nonspur": "Non-boundary & Non-spurious", "nonconf_spur": "Non-boundary & Spurious"}
ORDER = ["conf_nonspur", "conf_spur", "nonconf_nonspur", "nonconf_spur"]

for sigma4 in sigma4_list:
    history = run_experiment(n=10000, p_spurious_23=BASE_P23, p_spurious_4=p4,
                             sigma_23=BASE_SIGMA23, sigma_4=sigma4,
                             batch_size=128, num_epochs=10, lr=0.5, seed=0)
    epochs = np.arange(len(history["all"]))

    plt.figure(figsize=(4, 4))
    for k in ORDER:
        plt.plot(epochs, history[k], linewidth=2, linestyle="-", color=COLORS[k], label=LABELS[k])
    plt.xlabel("Training epochs", fontsize=LABEL_FONT)
    plt.ylabel("Average training loss", fontsize=LABEL_FONT)
    ax = plt.gca()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))
    ax.tick_params(axis="both", which="major", labelsize=TICK_FONT)
    plt.tight_layout()

    # one shared legend, on the rightmost panel only
    if sigma4 == sigma4_list[-1]:
        handles = [Patch(facecolor=COLORS[k], edgecolor="none") for k in ORDER]
        ax.legend(handles, [LABELS[k] for k in ORDER], loc="upper right",
                  fontsize=LEGEND_FONT, frameon=True, framealpha=0.9,
                  handlelength=1.1, handletextpad=0.4, labelspacing=0.25, borderpad=0.25)

    fname = f"img_results-2-x4/loss_p23_{BASE_P23}_s23_{BASE_SIGMA23}_p4_{p4}_s4_{sigma4}.pdf"
    plt.savefig(fname, bbox_inches="tight")
    plt.close()
    print(f"wrote {fname}")
